# ..Read ME...

This convert file.mat to csv 

Licence @ P' Jumbo


In [ ]:
from scipy.io import loadmat
import numpy as np
import pandas as pd
import os   

mat_dir = '../Results/matfile/2025/305/'# 'Mat_files/'
station = 'CM013160'
matfilename = f'{mat_dir}{station}.mat'



matfile = loadmat(matfilename)
# [SOD,sys,PRN,STEC,VTEC,ROTI]

data = matfile['CM01'][0,0]
fdata = data.dtype.names
# print(fdata)

# print(data['BDS'][0,0].dtype.names)
# print(data['BDS'][0,0]['stec'])
# print(data[fdata[0]][0,0])

date = data['GPS'][0,0]['date'][0]

year = date[0]
mth = date[1]
dt = date[2]

T_list = [None]*(len(fdata)-1)

for i, sys in enumerate(fdata[:-1]):
    print(sys)
    subdata = data[sys].ravel()[0]
    # print(subdata.dtype.names)
    time = subdata['ind'].ravel()
    nanmask = np.isfinite(time)
    SOD = sorted(list(set(time[nanmask]-1)))  # MATLAB is 1-based; convert to 0-based
    SOD = np.array(SOD).astype(int)
    stec = pd.DataFrame({f'stec_{i+1}': subdata['stec'][SOD,i] for i in range(subdata['stec'].shape[1])})
    vtec = pd.DataFrame({f'vtec_{i+1}': subdata['vtec'][SOD,i] for i in range(subdata['vtec'].shape[1])})
    roti = pd.DataFrame({f'roti_{i+1}': subdata['roti'][SOD,i] for i in range(subdata['roti'].shape[1])})
    ipplat = pd.DataFrame({f'ipplat_{i+1}': subdata['ipplat'][SOD,i] for i in range(subdata['ipplat'].shape[1])})
    ipplon = pd.DataFrame({f'ipplon_{i+1}': subdata['ipplon'][SOD,i] for i in range(subdata['ipplon'].shape[1])})
    nPRN = pd.DataFrame({'nPRN': np.ones(len(SOD),dtype=int) * stec.shape[1]})
    
    system = pd.DataFrame({'system': [sys]*SOD.shape[0]})

    SOD = pd.DataFrame({'SOD':SOD+1}) # Convert back to 1-based for output consistency
    T = pd.concat([system, SOD, nPRN, vtec, stec, roti, ipplat, ipplon], axis=1)
    # print(T.head()) # inspect the first few rows of the DataFrame
    T_list[i] = T # Store the DataFrame in the list

print(*T_list, sep='\n\n')


folder_name = f"../Results/{station}_csv"
os.makedirs(folder_name, exist_ok=True)


for T, sys in zip(T_list, fdata[:-1]):
    
    file_path = f"{folder_name}/{station}_{sys}.csv"
    T.to_csv(file_path, index=False)

BDS
GAL
GPS
QZS
GLO
     system    SOD  nPRN    vtec_1    vtec_2    vtec_3  vtec_4    vtec_5  \
0       BDS      1    41 -1.874190 -4.408097 -3.219744     NaN -7.497539   
1       BDS     31    41 -1.777693 -4.334075 -3.060713     NaN -7.397922   
2       BDS     61    41 -1.699485 -4.211497 -2.943961     NaN -7.213870   
3       BDS     91    41 -1.593945 -4.150078 -2.903175     NaN -7.122215   
4       BDS    121    41 -1.460539 -4.025674 -2.778394     NaN -6.998552   
...     ...    ...   ...       ...       ...       ...     ...       ...   
2874    BDS  86221    41 -3.107416 -6.814439 -5.869316     NaN -8.849717   
2875    BDS  86251    41 -2.994851 -6.691345 -5.719305     NaN -8.769404   
2876    BDS  86281    41 -2.909008 -6.648488 -5.641363     NaN -8.682131   
2877    BDS  86311    41 -2.758820 -6.571173 -5.471387     NaN -8.621527   
2878    BDS  86341    41 -2.699074 -6.382635 -5.359800     NaN -8.519908   

        vtec_6  vtec_7  ...  ipplon_32  ipplon_33   ipplon_34   ipp